In [8]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
from src.model.datasetbuilder.dataset_builder import DatasetBuilder

In [9]:
builder = DatasetBuilder()
tmt_agg = builder._load_tmt_aggregated()

print(f"Shape: {tmt_agg.shape}")
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame({'column': tmt_agg.columns}))

Shape: (368, 105)


,column
0,subject_id
1,area_difference_from_ideal_PART_A
2,area_difference_from_ideal_PART_B
3,average_duration_PART_A
4,average_duration_PART_B
5,correct_targets_touches_PART_A
6,correct_targets_touches_PART_B
7,distance_difference_from_ideal_PART_A
8,distance_difference_from_ideal_PART_B
9,hesitation_avg_speed_PART_A


In [10]:
from analysis.scripts.figures.shap_common import FEATURE_LABELS

# Build expected feature set: raw cols + B_A_ratio derived cols
raw_cols = [c for c in tmt_agg.columns if c != "subject_id"]

base_names_A = {c.replace("_PART_A", "") for c in raw_cols if c.endswith("_PART_A")}
base_names_B = {c.replace("_PART_B", "") for c in raw_cols if c.endswith("_PART_B")}
ratio_cols = [f"{b}_B_A_ratio" for b in base_names_A & base_names_B]

all_expected = set(raw_cols) | set(ratio_cols)
labeled = set(FEATURE_LABELS.keys())

missing = sorted(all_expected - labeled)
stale   = sorted(labeled - all_expected)

print(f"Total expected features : {len(all_expected)}")
print(f"Covered in FEATURE_LABELS: {len(labeled & all_expected)}")
print(f"\nMissing from FEATURE_LABELS ({len(missing)}):")
for c in missing:
    print(f"  {c}")

print(f"\nStale FEATURE_LABELS keys ({len(stale)}) — not in tmt_agg or ratios:")
for c in stale:
    print(f"  {c}")

Total expected features : 156
Covered in FEATURE_LABELS: 105

Missing from FEATURE_LABELS (51):
  correct_targets_touches_B_A_ratio
  correct_targets_touches_PART_A
  correct_targets_touches_PART_B
  hesitation_ratio_B_A_ratio
  hesitation_ratio_PART_A
  hesitation_ratio_PART_B
  non_cut_area_difference_from_ideal_B_A_ratio
  non_cut_area_difference_from_ideal_PART_A
  non_cut_area_difference_from_ideal_PART_B
  non_cut_average_duration_B_A_ratio
  non_cut_average_duration_PART_A
  non_cut_average_duration_PART_B
  non_cut_distance_difference_from_ideal_B_A_ratio
  non_cut_distance_difference_from_ideal_PART_A
  non_cut_distance_difference_from_ideal_PART_B
  non_cut_hesitation_avg_speed_B_A_ratio
  non_cut_hesitation_avg_speed_PART_A
  non_cut_hesitation_avg_speed_PART_B
  non_cut_hesitation_distance_B_A_ratio
  non_cut_hesitation_distance_PART_A
  non_cut_hesitation_distance_PART_B
  non_cut_hesitation_ratio_B_A_ratio
  non_cut_hesitation_ratio_PART_A
  non_cut_hesitation_ratio_PART_

In [11]:
FEATURE_DEFINITIONS = {
    "area_difference_from_ideal":     {"paper_name": "Area difference from ideal",     "description": "Difference between the area of the actual path and the ideal path."},
    "correct_targets_touches":        {"paper_name": "Correct touches",                "description": "Number of correct target touches"},
    "distance_difference_from_ideal": {"paper_name": "Distance difference from ideal", "description": "Difference between the traveled path length and the length of the ideal straight line connecting to the next target."},
    "max_duration":                   {"paper_name": "Hesitation max duration",        "description": "Longest hesitation duration during the trial"},
    "average_duration":               {"paper_name": "Hesitation average duration",               "description": "Mean duration of individual hesitation periods detected in the trial."},
    "hesitation_avg_speed":           {"paper_name": "Hesitation average speed",       "description": "Average hesitation speed."},
    "hesitation_distance":            {"paper_name": "Hesitation distance",            "description": "Total distance traveled by the cursor in hesitation"},
    "hesitation_ratio":               {"paper_name": "Hesitation ratio",               "description": "Proportion of time in hesitation relative to total travel time: hesitation_time / (travel_time + hesitation_time)."},
    "hesitation_time":                {"paper_name": "Hesitation time",                "description": "Total time spent in the hesitation state during the trial."},
    "inter_target_time":              {"paper_name": "Inter-target time",              "description": "Sum of time elapsed between successive target touches."},
    "intra_target_time":              {"paper_name": "Intra-target time",              "description": "Sum of time spent within the area of a target before leaving it."},
    "mean_speed":                     {"paper_name": "Mean speed",                     "description": "Average cursor speed."},
    "number_of_crosses":              {"paper_name": "Number of crosses",              "description": "Number of times the cursor trajectory crosses itself, excluding intersections between temporally adjacent segments."},
    "peak_speed":                     {"paper_name": "Peak speed",                     "description": "Maximum cursor speed observed."},
    "rt":                             {"paper_name": "Time in trial",                  "description": "Total duration of the trial."},
    "search_avg_speed":               {"paper_name": "Search average speed",           "description": "Average cursor speed during search movements."},
    "search_distance":                {"paper_name": "Search distance",               "description": "Total distance traveled during search movements."},
    "search_time":                    {"paper_name": "Search time",                    "description": "Total time spent in the search state."},
    "state_transitions":              {"paper_name": "State transitions",              "description": "Total number of transitions for one state (hesitation, search or travel) to another"},
    "std_speed":                      {"paper_name": "SD speed",                       "description": "Standard deviation of the cursor speed."},
    "total_distance":                 {"paper_name": "Total distance",                 "description": "Total distance traveled by the cursor."},
    "total_hesitations":              {"paper_name": "Hesitations",                    "description": "Count of distinct hesitation periods detected in the trial."},
    "travel_avg_speed":               {"paper_name": "Travel average speed",           "description": "Average cursor speed during travel segments."},
    "travel_distance":                {"paper_name": "Travel distance",               "description": "Total distance traveled during travel segments."},
    "travel_time":                    {"paper_name": "Travel time",                    "description": "Total time spent during travel segments."},
    "wrong_targets_touches":          {"paper_name": "Wrong target touches",           "description": "Count of cursor touches on incorrect targets (not the next expected in the sequence)."},
}

print(f"Total base features: {len(FEATURE_DEFINITIONS)}")

Total base features: 26


In [12]:
def expand_feature_definitions(feature_defs: dict) -> list[str]:
    cols = []
    for feat in feature_defs:
        cols.append(f"{feat}_PART_A")
        cols.append(f"{feat}_PART_B")
        cols.append(f"non_cut_{feat}_PART_A")
        cols.append(f"non_cut_{feat}_PART_B")
    return cols

all_104_metrics = expand_feature_definitions(FEATURE_DEFINITIONS)
print(f"Total metrics: {len(all_104_metrics)}")
print(f"Matches tmt_agg columns: {set(all_104_metrics) == set(tmt_agg.columns) - {'subject_id'}}")

Total metrics: 104
Matches tmt_agg columns: True


In [13]:
import pandas as pd

df_features = pd.DataFrame([
    {"Code name": key, "Feature": val["paper_name"], "Description": val["description"]}
    for key, val in FEATURE_DEFINITIONS.items()
])

df_features.to_csv("feature_definitions.csv", index=False)
df_features

,Code name,Feature,Description
0,area_difference_from_ideal,Area difference from ideal,Difference between the area of the actual path...
1,correct_targets_touches,Correct touches,Number of correct target touches
2,distance_difference_from_ideal,Distance difference from ideal,Difference between the traveled path length an...
3,max_duration,Hesitation max duration,Longest hesitation duration during the trial
4,average_duration,Hesitation average duration,Mean duration of individual hesitation periods...
5,hesitation_avg_speed,Hesitation average speed,Average hesitation speed.
6,hesitation_distance,Hesitation distance,Total distance traveled by the cursor in hesit...
7,hesitation_ratio,Hesitation ratio,Proportion of time in hesitation relative to t...
8,hesitation_time,Hesitation time,Total time spent in the hesitation state durin...
9,inter_target_time,Inter-target time,Sum of time elapsed between successive target ...
